In [17]:
import pandas as pd
import chemsource
import os
import sys
import time
import asyncio
import numpy as np


sys.path.append(os.path.abspath("../src"))
from harmonization import (
    harmonize_automated_classification,
    harmonize_manual_classification,
)


In [3]:
def classification_to_bits(classification):
    categories = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
    bits = ["1" if category in classification else "0" for category in categories]

    if len(set(classification) - set(categories)) > 0:
        unknown_categories = ",".join(set(classification) - set(categories))
        return "".join(bits)+f"_UNKNOWN({unknown_categories})"
    return "".join(bits)

def bits_to_classification(bits):
    categories = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]
    classification = [categories[i] for i in range(len(categories)) if bits[i] == "1"]
    return classification

reasoning_data_out_path_non_medical = "../data/random_sample_reruns/reasoning_non_medical_random_sample_reruns.csv"
reasoning_data_out_path_medical = "../data/random_sample_reruns/reasoning_medical_random_sample_reruns.csv"

data_out_path_non_medical = "../data/random_sample_reruns/non_medical_random_sample_reruns_NEW_RETRY.csv"
data_out_path_medical = "../data/random_sample_reruns/medical_random_sample_reruns.csv"


In [ ]:
classified_drug_library_data_path = "../data/drug_library/validation_data_classified_all_3_methods.csv"

drug_library_text = pd.read_csv(classified_drug_library_data_path)
drug_library_text["FEATURE_ID"] = drug_library_text.index
drug_library_text = drug_library_text[["FEATURE_ID", 
    "name_used", 
    "text"]].rename(columns={
    "name_used": "NAME",
    "text": "TEXT"})

id_to_name = dict(zip(drug_library_text["FEATURE_ID"].astype(str).tolist(), 
                      drug_library_text["NAME"].tolist()))

In [42]:
def compute_frequencies(data_path):
    data = pd.read_csv(data_path, dtype=str)
    data = data.rename(columns=id_to_name)
    data = data.map(lambda x: np.array([int(item) for item in list(str(x))]))
    data = data.sum(axis=0)
    column_names = ["MEDICAL", "FOOD", "INDUSTRIAL", "PERSONAL CARE", "ENDOGENOUS", "INFO"]

    expanded_features = pd.DataFrame(data.tolist(), index=data.index)
    expanded_features.columns = column_names

    
    return expanded_features


In [46]:
compute_frequencies(data_out_path_medical).to_csv("../data/random_sample_reruns/results/medical_reruns_default_model.csv")
compute_frequencies(data_out_path_non_medical).to_csv("../data/random_sample_reruns/results/non_medical_reruns_default_model.csv")
compute_frequencies(reasoning_data_out_path_non_medical).to_csv("../data/random_sample_reruns/results/non_medical_reruns_reasoning_model.csv")